<a href="https://colab.research.google.com/github/JuanFerMC/TeoriaDelLenguaje-LABS/blob/main/Arbol_de_derivaci%C3%B3n_Gr%C3%A1fico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Informe Práctica #5**
* **Juan Fernando Mona Cano | juanf.mona@udea.edu.co**
---
## **Introduccion:**
El presente informe detalla el diseño y la lógica detrás de la implementación de una herramienta de software capaz de analizar y visualizar derivaciones para una gramática indepentiente del contexto `(GIC)` específica.

## **Objetivo:**
Implementar un analizador sintáctico descendente que utilice la técnica de backtracking completo mediante generadores de Python para encontrar derivaciones válidas y visualizar su árbol sintáctico correspondiente.

## **Cuerpo**
---
La gramática proporcionada, denotada por G=(V,Σ,R,S), se define como sigue:

  * **Símbolos No Terminales (V): {S,A,B}**
  * **Símbolos Terminales (Σ): {a,b}**
  * **Símbolo Inicial: S**
  * **Símbolo Vacío (λ): λ (representa la cadena vacía)**
  * **Reglas de Producción (R):**
    * S	→ aAb | aBb | $\lambda$
    * A	→ aAb | a
    * B	→ aBb | b | $\lambda$

El programa se divide en tres componentes fundamentales: representación de datos, lógica de análisis y visualización.

Se utiliza una estructura de árbol n-ario donde cada instancia de la clase representa un símbolo (terminal o no terminal).

  * **Atributos:** simbolo (el valor del nodo) y hijos (lista de referencias a otros objetos NodoArbol).

  * **Métodos:** agregar_hijo permite la construcción dinámica del árbol durante el proceso de derivación.

A diferencia de un parser recursivo simple que se detiene al primer error, esta implementación utiliza generadores de Python (yield). Esto permite:

  * **Búsqueda Exhaustiva:** Explorar todas las producciones posibles para un símbolo.

  * **Manejo de Posiciones:** Cada función de derivación (derivar_S, derivar_A, derivar_B) recibe la cadena y la posición actual, y devuelve un par (nodo, nueva_posicion).

  * **Recursividad:** Las funciones se llaman a sí mismas para resolver sub-problemas (como A→aAb).

### Ejemplo de flujo para S:
Cuando se evalúa S, el programa intenta la producción λ. Si no agota la cadena, retrocede (backtrack) y prueba con aAb. Si esta falla en algún punto, retrocede nuevamente para intentar con aBb.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "ipywidgets"], check=True)
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output

import math
import html as html_lib

class Nodo:
    def __init__(self, simbolo):
        self.simbolo = simbolo
        self.hijos = []

    def es_terminal(self):
        return self.simbolo in ('a', 'b', 'λ')

    def __repr__(self):
        return f"Nodo({self.simbolo})"


def derivar_S(w, pos):
    if pos == len(w):
        n = Nodo("S"); n.hijos.append(Nodo("λ")); yield n, pos
    if pos < len(w) and w[pos] == 'a':
        for hA, p1 in derivar_A(w, pos + 1):
            if p1 < len(w) and w[p1] == 'b':
                n = Nodo("S")
                n.hijos += [Nodo("a"), hA, Nodo("b")]
                yield n, p1 + 1
        for hB, p2 in derivar_B(w, pos + 1):
            if p2 < len(w) and w[p2] == 'b':
                n = Nodo("S")
                n.hijos += [Nodo("a"), hB, Nodo("b")]
                yield n, p2 + 1


def derivar_A(w, pos):
    if pos >= len(w) or w[pos] != 'a':
        return
    for hA, p in derivar_A(w, pos + 1):
        if p < len(w) and w[p] == 'b':
            n = Nodo("A")
            n.hijos += [Nodo("a"), hA, Nodo("b")]
            yield n, p + 1
    n = Nodo("A"); n.hijos.append(Nodo("a")); yield n, pos + 1


def derivar_B(w, pos):
    n = Nodo("B"); n.hijos.append(Nodo("λ")); yield n, pos
    if pos < len(w) and w[pos] == 'b':
        n = Nodo("B"); n.hijos.append(Nodo("b")); yield n, pos + 1
    if pos < len(w) and w[pos] == 'a':
        for hB, p in derivar_B(w, pos + 1):
            if p < len(w) and w[p] == 'b':
                n = Nodo("B")
                n.hijos += [Nodo("a"), hB, Nodo("b")]
                yield n, p + 1


def parsear(cadena):
    for nodo, pos in derivar_S(cadena, 0):
        if pos == len(cadena):
            return nodo
    return None

NODE_W   = 46    # ancho de cada nodo
NODE_H   = 36    # alto de cada nodo
H_GAP    = 14    # espacio mínimo horizontal entre nodos hermanos
V_GAP    = 64    # espacio vertical entre niveles
PADDING  = 30    # margen exterior


def calcular_ancho_subárbol(nodo):
    """Calcula el ancho mínimo necesario para el subárbol del nodo."""
    if not nodo.hijos:
        return NODE_W
    total = sum(calcular_ancho_subárbol(h) for h in nodo.hijos)
    total += H_GAP * (len(nodo.hijos) - 1)
    return max(total, NODE_W)


def asignar_posiciones(nodo, x_centro, y, posiciones):
    """Asigna coordenadas (cx, cy) a cada nodo recursivamente."""
    posiciones[id(nodo)] = (x_centro, y)
    if not nodo.hijos:
        return

    anchos = [calcular_ancho_subárbol(h) for h in nodo.hijos]
    total_ancho = sum(anchos) + H_GAP * (len(nodo.hijos) - 1)
    x_inicio = x_centro - total_ancho / 2

    for hijo, ancho in zip(nodo.hijos, anchos):
        cx_hijo = x_inicio + ancho / 2
        asignar_posiciones(hijo, cx_hijo, y + V_GAP + NODE_H, posiciones)
        x_inicio += ancho + H_GAP


def profundidad_arbol(nodo):
    if not nodo.hijos:
        return 1
    return 1 + max(profundidad_arbol(h) for h in nodo.hijos)


def recolectar_nodos(nodo):
    yield nodo
    for h in nodo.hijos:
        yield from recolectar_nodos(h)

COLORES = {
    "S": ("#1a1a2e", "#e94560", "#ffffff"),   # fondo oscuro, borde coral, texto blanco
    "A": ("#162447", "#1f9daf", "#e0f7fa"),   # azul marino, cian, texto claro
    "B": ("#1b2838", "#f0a500", "#fff8e1"),   # azul oscuro, ámbar, texto cálido
    "a": ("#0f3460", "#53c0f0", "#ffffff"),   # terminal 'a'
    "b": ("#3d0c02", "#ff6b6b", "#ffffff"),   # terminal 'b'
    "λ": ("#1a1a1a", "#888888", "#cccccc"),   # lambda gris
}

def color_nodo(simbolo):
    return COLORES.get(simbolo, ("#2d2d2d", "#aaaaaa", "#ffffff"))


def generar_svg(raiz):
    posiciones = {}
    ancho_total = calcular_ancho_subárbol(raiz)
    prof = profundidad_arbol(raiz)

    svg_w = max(ancho_total + 2 * PADDING, 340)
    svg_h = prof * (NODE_H + V_GAP) + PADDING * 2

    cx_raiz = svg_w / 2
    asignar_posiciones(raiz, cx_raiz, PADDING + NODE_H / 2, posiciones)

    lineas = []
    nodos_svg = []

    for nodo in recolectar_nodos(raiz):
        cx, cy = posiciones[id(nodo)]
        for hijo in nodo.hijos:
            hx, hy = posiciones[id(hijo)]
            # Línea curva entre padre e hijo
            ctrl_y = (cy + hy) / 2
            lineas.append(
                f'<path d="M {cx:.1f} {cy + NODE_H/2:.1f} '
                f'C {cx:.1f} {ctrl_y:.1f}, {hx:.1f} {ctrl_y:.1f}, '
                f'{hx:.1f} {hy - NODE_H/2:.1f}" '
                f'stroke="#4a4a6a" stroke-width="1.8" fill="none" '
                f'stroke-dasharray="none" opacity="0.7"/>'
            )

    for nodo in recolectar_nodos(raiz):
        cx, cy = posiciones[id(nodo)]
        fondo, borde, texto_color = color_nodo(nodo.simbolo)
        sym = html_lib.escape(nodo.simbolo)
        rx = NODE_W / 2
        ry = NODE_H / 2

        es_no_terminal = nodo.simbolo in ("S", "A", "B")

        # Sombra
        nodos_svg.append(
            f'<ellipse cx="{cx + 2:.1f}" cy="{cy + 3:.1f}" '
            f'rx="{rx:.0f}" ry="{ry:.0f}" '
            f'fill="rgba(0,0,0,0.35)"/>'
        )

        if es_no_terminal:
            nodos_svg.append(
                f'<ellipse cx="{cx:.1f}" cy="{cy:.1f}" '
                f'rx="{rx:.0f}" ry="{ry:.0f}" '
                f'fill="{fondo}" stroke="{borde}" stroke-width="2.2"/>'
            )
        else:
            x0 = cx - rx
            y0 = cy - ry
            nodos_svg.append(
                f'<rect x="{x0:.1f}" y="{y0:.1f}" '
                f'width="{NODE_W}" height="{NODE_H}" '
                f'rx="8" ry="8" '
                f'fill="{fondo}" stroke="{borde}" stroke-width="2.2"/>'
            )

        nodos_svg.append(
            f'<text x="{cx:.1f}" y="{cy:.1f}" '
            f'text-anchor="middle" dominant-baseline="central" '
            f'font-family="\'Courier New\', monospace" '
            f'font-size="15" font-weight="bold" fill="{texto_color}">'
            f'{sym}</text>'
        )

    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" '
        f'width="{svg_w:.0f}" height="{svg_h:.0f}" '
        f'viewBox="0 0 {svg_w:.0f} {svg_h:.0f}" '
        f'style="background:#0d0d1a; border-radius:12px; display:block; margin:auto;">'
        f'<defs>'
        f'<radialGradient id="bg_grad" cx="50%" cy="0%" r="80%">'
        f'<stop offset="0%" stop-color="#1a1a3e"/>'
        f'<stop offset="100%" stop-color="#0d0d1a"/>'
        f'</radialGradient>'
        f'</defs>'
        f'<rect width="{svg_w:.0f}" height="{svg_h:.0f}" fill="url(#bg_grad)" rx="12"/>'
    )
    svg += "".join(lineas)
    svg += "".join(nodos_svg)
    svg += "</svg>"
    return svg, int(svg_w), int(svg_h)

ESTILOS_BASE = """
<style>
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&family=Syne:wght@700;800&display=swap');

.app-container {
    background: linear-gradient(135deg, #0d0d1a 0%, #1a0a2e 50%, #0a1628 100%);
    border-radius: 16px;
    padding: 28px 32px 32px;
    font-family: 'JetBrains Mono', monospace;
    color: #c8c8e8;
    max-width: 820px;
    margin: 0 auto;
    box-shadow: 0 8px 48px rgba(0,0,0,0.6), inset 0 1px 0 rgba(255,255,255,0.05);
    border: 1px solid rgba(100, 80, 200, 0.2);
}

.app-title {
    font-family: 'Syne', sans-serif;
    font-size: 22px;
    font-weight: 800;
    color: #e94560;
    letter-spacing: 0.04em;
    margin-bottom: 4px;
}

.app-subtitle {
    font-size: 11px;
    color: #6060a0;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    margin-bottom: 22px;
}

.grammar-box {
    background: rgba(255,255,255,0.03);
    border: 1px solid rgba(100, 80, 200, 0.25);
    border-left: 3px solid #e94560;
    border-radius: 8px;
    padding: 14px 18px;
    margin-bottom: 22px;
    font-size: 13px;
    line-height: 2;
}

.grammar-rule { color: #c8c8e8; }
.grammar-nt   { color: #1f9daf; font-weight: bold; }
.grammar-arr  { color: #6060a0; }
.grammar-t    { color: #f0a500; }
.grammar-lam  { color: #888; font-style: italic; }

.result-ok {
    background: rgba(31,157,175,0.12);
    border: 1px solid rgba(31,157,175,0.4);
    border-radius: 8px;
    padding: 12px 18px;
    color: #1f9daf;
    font-size: 13px;
    margin-bottom: 16px;
    display: flex;
    align-items: center;
    gap: 10px;
}

.result-err {
    background: rgba(233,69,96,0.12);
    border: 1px solid rgba(233,69,96,0.4);
    border-radius: 8px;
    padding: 12px 18px;
    color: #e94560;
    font-size: 13px;
    margin-bottom: 16px;
    display: flex;
    align-items: center;
    gap: 10px;
}

.tree-label {
    font-size: 11px;
    color: #6060a0;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    margin-bottom: 10px;
}

.legend {
    display: flex;
    gap: 16px;
    flex-wrap: wrap;
    margin-top: 18px;
    padding-top: 14px;
    border-top: 1px solid rgba(255,255,255,0.06);
    font-size: 11px;
}

.legend-item { display: flex; align-items: center; gap: 6px; }
.legend-dot  {
    width: 12px; height: 12px; border-radius: 50%;
    border: 2px solid;
    display: inline-block;
}
</style>
"""

def html_gramatica():
    return """
    <div class="grammar-box">
        <span class="grammar-nt">S</span>
        <span class="grammar-arr"> → </span>
        <span class="grammar-nt">a</span><span class="grammar-t">A</span><span class="grammar-nt">b</span>
        <span class="grammar-arr"> | </span>
        <span class="grammar-nt">a</span><span class="grammar-t">B</span><span class="grammar-nt">b</span>
        <span class="grammar-arr"> | </span>
        <span class="grammar-lam">λ</span>
        <br>
        <span class="grammar-t">A</span>
        <span class="grammar-arr"> → </span>
        <span class="grammar-nt">a</span><span class="grammar-t">A</span><span class="grammar-nt">b</span>
        <span class="grammar-arr"> | </span>
        <span class="grammar-nt">a</span>
        <br>
        <span class="grammar-t">B</span>
        <span class="grammar-arr"> → </span>
        <span class="grammar-nt">a</span><span class="grammar-t">B</span><span class="grammar-nt">b</span>
        <span class="grammar-arr"> | </span>
        <span class="grammar-nt">b</span>
        <span class="grammar-arr"> | </span>
        <span class="grammar-lam">λ</span>
    </div>
    """

def html_leyenda():
    return """
    <div class="legend">
        <span style="color:#6060a0; font-size:11px; margin-right:4px;">LEYENDA:</span>
        <span class="legend-item">
            <span class="legend-dot" style="background:#1a1a2e; border-color:#e94560;"></span>
            <span style="color:#e94560">S</span> — símbolo inicial
        </span>
        <span class="legend-item">
            <span class="legend-dot" style="background:#162447; border-color:#1f9daf;"></span>
            <span style="color:#1f9daf">A</span> — no terminal A
        </span>
        <span class="legend-item">
            <span class="legend-dot" style="background:#1b2838; border-color:#f0a500;"></span>
            <span style="color:#f0a500">B</span> — no terminal B
        </span>
        <span class="legend-item">
            <span class="legend-dot" style="background:#0f3460; border-color:#53c0f0; border-radius:2px;"></span>
            <span style="color:#53c0f0">a</span> — terminal
        </span>
        <span class="legend-item">
            <span class="legend-dot" style="background:#3d0c02; border-color:#ff6b6b; border-radius:2px;"></span>
            <span style="color:#ff6b6b">b</span> — terminal
        </span>
        <span class="legend-item">
            <span class="legend-dot" style="background:#1a1a1a; border-color:#888; border-radius:2px;"></span>
            <span style="color:#888">λ</span> — cadena vacía
        </span>
    </div>
    """


def construir_interfaz():
    campo = widgets.Text(
        placeholder='Ej: aab, aaabb, ab, abb, aabb ...',
        layout=widgets.Layout(width='400px', height='38px'),
        style={'description_width': '0px'},
    )
    campo.add_class('input-word')

    boton = widgets.Button(
        description='Analizar  ▶',
        button_style='',
        layout=widgets.Layout(width='130px', height='38px'),
    )

    boton_limpiar = widgets.Button(
        description='Limpiar',
        button_style='',
        layout=widgets.Layout(width='90px', height='38px'),
    )

    salida = widgets.Output()

    cabecera = widgets.HTML(value=f"""
    {ESTILOS_BASE}
    <div class="app-container">
        <div class="app-title">⬡ Árbol de Derivación</div>
        <div class="app-subtitle">Analizador sintáctico · Gramática libre de contexto</div>
        {html_gramatica()}
    </div>
    """)

    estilos_widgets = widgets.HTML(value="""
    <style>
    .widget-text input[type="text"] {
        background: rgba(255,255,255,0.04) !important;
        border: 1px solid rgba(100,80,200,0.4) !important;
        border-radius: 8px !important;
        color: #e0e0ff !important;
        font-family: 'JetBrains Mono', monospace !important;
        font-size: 14px !important;
        padding: 0 14px !important;
        outline: none !important;
        transition: border-color 0.2s !important;
    }
    .widget-text input[type="text"]:focus {
        border-color: #e94560 !important;
        box-shadow: 0 0 0 2px rgba(233,69,96,0.2) !important;
    }
    .widget-button button {
        background: linear-gradient(135deg, #e94560, #c73050) !important;
        border: none !important;
        border-radius: 8px !important;
        color: #fff !important;
        font-family: 'JetBrains Mono', monospace !important;
        font-size: 13px !important;
        font-weight: bold !important;
        cursor: pointer !important;
        transition: opacity 0.15s !important;
    }
    .widget-button button:hover { opacity: 0.85 !important; }
    </style>
    """)

    etiqueta_input = widgets.HTML(
        value='<span style="font-family:\'JetBrains Mono\',monospace; '
              'color:#6060a0; font-size:12px; letter-spacing:0.1em; '
              'text-transform:uppercase; line-height:38px; padding-right:10px;">'
              'Palabra:</span>'
    )

    fila_entrada = widgets.HBox(
        [etiqueta_input, campo, boton, boton_limpiar],
        layout=widgets.Layout(
            align_items='center',
            gap='8px',
            padding='0 0 8px 0',
        )
    )

    # ── Lógica de análisis ───────────────────────────────────────────
    def analizar(_=None):
        palabra = campo.value.strip()
        with salida:
            clear_output(wait=True)

            if palabra and not all(c in 'ab' for c in palabra):
                display(HTML(f"""
                {ESTILOS_BASE}
                <div class="app-container" style="padding:20px 28px;">
                    <div class="result-err">
                        <span style="font-size:18px;">⚠</span>
                        <span>La palabra <b>"{html_lib.escape(palabra)}"</b>
                        contiene símbolos no válidos.<br>
                        Solo se permiten los caracteres <b>a</b> y <b>b</b>.</span>
                    </div>
                </div>
                """))
                return

            raiz = parsear(palabra)
            muestra = f'"{html_lib.escape(palabra)}"' if palabra else '"λ" (cadena vacía)'

            if raiz is None:
                display(HTML(f"""
                {ESTILOS_BASE}
                <div class="app-container" style="padding:20px 28px;">
                    <div class="result-err">
                        <span style="font-size:18px;">✗</span>
                        <span>La palabra <b>{muestra}</b>
                        <b>NO es aceptada</b> por la gramática.</span>
                    </div>
                </div>
                """))
                return

            svg_str, w, h = generar_svg(raiz)
            regla = detectar_regla(raiz)

            display(HTML(f"""
            {ESTILOS_BASE}
            <div class="app-container">
                <div class="result-ok">
                    <span style="font-size:18px;">✓</span>
                    <span>La palabra <b>{muestra}</b> es <b>aceptada</b>.
                    &nbsp;·&nbsp; Regla aplicada: <code style="color:#f0a500">{regla}</code></span>
                </div>
                <div class="tree-label">Árbol de derivación</div>
                <div style="overflow-x:auto; padding:4px 0;">
                    {svg_str}
                </div>
                {html_leyenda()}
            </div>
            """))

    def limpiar(_):
        campo.value = ''
        with salida:
            clear_output()

    def on_enter(change):
        if change.get('new', '').endswith('\n') or True:
            pass  # handled by observe on submit

    campo.on_submit(lambda _: analizar())
    boton.on_click(analizar)
    boton_limpiar.on_click(limpiar)
    display(cabecera)
    display(estilos_widgets)
    display(fila_entrada)
    display(salida)

def detectar_regla(raiz):
    """Detecta qué producción de S se aplicó."""
    hijos = [h.simbolo for h in raiz.hijos]
    if hijos == ['λ']:
        return 'S → λ'
    if 'A' in hijos:
        return 'S → aAb'
    if 'B' in hijos:
        return 'S → aBb'
    return 'S → ?'

construir_interfaz()

HTML(value='\n    \n<style>\n@import url(\'https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;70…

HTML(value='\n    <style>\n    .widget-text input[type="text"] {\n        background: rgba(255,255,255,0.04) !…

Output()

# **Conclusiones**

La realización de esta práctica ha permitido obtener las siguientes conclusiones clave:

  * Se ha demostrado cómo conceptos abstractos como "gramáticas libres de contexto" y "reglas de producción" se traducen directamente en estructuras de datos jerárquicas (árboles) y lógica de flujo de control (funciones recursivas) en la programación.

  * El diseño del parser ha resaltado que la forma de las producciones afecta directamente a la complejidad de la implementación. Una gramática ambigua requiere una lógica de backtracking más costosa o técnicas de preprocesamiento, a diferencia de una gramática determinista.

  * El uso de una estructura de árbol es fundamental no solo para el análisis interno, sino también para proporcionar una visualización intuitiva de cómo se estructuran las oraciones de un lenguaje según sus reglas.

  * La capacidad de detectar entradas inválidas es tan crucial como la capacidad de procesar las válidas, lo que refuerza la importancia de la fase de validación de sintaxis en cualquier sistema de interpretación de lenguaje.